# Random Forest: Stable Churn Prediction

**Family:** Supervised learning | **Level:** Intermediate | **Dataset:** `telco_churn` | **Runs on:** CPU, about 2 minute(s)

Average many randomised trees to cut variance, and use out-of-bag data as a free validation set.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- Why bagging reduces variance
- What random feature subsets add
- How out of bag error works
- Why impurity importance can mislead

**Prerequisites:**

- Linear regression notebook
- Train, validation and test splits
- Confusion matrix basics

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Subscription businesses lose revenue when customers leave, and winning a new customer costs more than keeping one.

**The problem.** Predict which telecom customers will churn from contract, billing and service information.

**Why it matters.** A ranked list of at risk customers lets a retention team spend a limited budget on the right people.

**Why machine learning.** Churn depends on many interacting attributes, and the history of past customers is available.

**Why this algorithm.** A forest keeps the flexibility of trees and removes most of their instability.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### Random Forests

- **Type:** foundational paper
- **Authors:** Breiman, L.
- **Year and venue:** 2001, Machine Learning 45(1):5-32
- **DOI or URL:** https://doi.org/10.1023/A:1010933404324
- **Verification:** Crossref 2026-11-17
- **Research objective:** Study ensembles of randomised trees and bound their generalisation error.
- **Methodology:** Bagging with a random subset of features at every split. The error bound depends on tree strength and correlation.
- **Dataset:** Several UCI benchmark datasets, compared with AdaBoost.
- **Main findings:** Forests are competitive with AdaBoost, and give out of bag error and variable importance.
- **Limitations:** The bound is loose in practice. Importance scores can mislead when features are correlated.

### Why do tree-based models still outperform deep learning on tabular data?

- **Type:** modern paper
- **Authors:** Grinsztajn, L., Oyallon, E. and Varoquaux, G.
- **Year and venue:** 2022, NeurIPS 2022 Datasets and Benchmarks; arXiv:2207.08815
- **DOI or URL:** https://arxiv.org/abs/2207.08815
- **Verification:** arXiv abstract page 2026-11-17
- **Research objective:** Benchmark tree models against neural networks on medium sized tabular data.
- **Methodology:** Large benchmark with random search budgets over 45 datasets as described in the paper.
- **Dataset:** Tabular benchmark collected from OpenML and other public sources.
- **Main findings:** Tree ensembles stayed ahead of the deep models on the studied datasets.
- **Limitations:** Medium sized data only, no very large datasets.

**Relationship to this notebook.** Breiman introduces random forests. Grinsztajn et al. benchmark tree ensembles on tabular data. We use one dataset only, so we cannot test their general claim. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Ensemble.** Train $B$ trees, tree $b$ on a bootstrap sample $\mathcal{D}_b$ and, at every split, on a random subset of $m$ features. Predict by averaging class probabilities:

$$\hat p(y=1 \mid x) = \frac1B \sum_{b=1}^{B} \hat p_b(y=1 \mid x)$$

**Variance of an average.** For $B$ identically distributed trees with variance $\sigma^2$ and pairwise correlation $\rho$:

$$\operatorname{Var}\Big(\frac1B\sum_b T_b\Big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

Random feature subsets lower $\rho$, bagging lowers the second term.

**Out of bag (OOB).** About 37 percent of rows are absent from each bootstrap sample. Predicting each row only with trees that did not see it gives an honest error estimate.

**Limitations.** Less interpretable than one tree, large models, impurity importance favours high cardinality and continuous features.

## 5. Dataset Introduction

**Telco customer churn**, IBM sample data from the repository `IBM/telco-customer-churn-on-icp4d` (code licence Apache-2.0, dataset terms not stated). 7043 customers, 20 features: demographics (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), account (`tenure` in months, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`) and services (`PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`). Target `Churn` (1 = customer left). `TotalCharges` has blank strings that become missing values. The file is downloaded once and checked against a SHA-256 hash. Limit: one snapshot, no time dimension. Duplicate rows are reported in the EDA but kept, because without an identifier we cannot tell whether they are different customers.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "random_forest"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("telco_churn")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. 

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.

- `TotalCharges` is missing for new customers, so the median imputer is fitted on training rows only.
- Churn is the minority class, so we select models on PR-AUC (area under the precision recall curve) instead of accuracy.

In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=False, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

Each tree overfits differently. Averaging cancels much of that noise while keeping the signal that all trees share.

### 8.2 Python implementation from scratch

We build a bagged forest of 40 trees from scikit-learn trees and compare with one tree. This shows the variance reduction directly.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
ytr = y_train.values
rng = np.random.default_rng(SEED)
trees = []
for b in range(40):
    idx = rng.integers(0, len(Xtr), len(Xtr))     # bootstrap rows
    t = DecisionTreeClassifier(max_features="sqrt", min_samples_leaf=3, random_state=b).fit(Xtr[idx], ytr[idx])
    trees.append(t)
forest_proba = np.mean([t.predict_proba(Xva)[:, 1] for t in trees], axis=0)
single_proba = trees[0].predict_proba(Xva)[:, 1]
print("validation ROC-AUC, single tree:", round(roc_auc_score(y_val, single_proba), 3))
print("validation ROC-AUC, forest of 40:", round(roc_auc_score(y_val, forest_proba), 3))

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `n_estimators` | Number of trees. More is rarely harmful. |
| `max_features` | Features tried at each split. |
| `min_samples_leaf` | Leaf size, smooths predictions. |
| `oob_score` | Compute out of bag score. |
| `class_weight` | Reweight classes. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "PR_AUC"
LOWER_IS_BETTER = False
SCORING = "average_precision"
SWEEP_PARAM = "model__max_features"
SWEEP_VALUES = [0.1, 0.2, 0.3, 0.5, 0.8]

def build_pipeline():
    return make_model_pipeline(X_train, RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=2, random_state=SEED), scale=False, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Classification metrics: accuracy, precision, recall, F1, ROC-AUC and PR-AUC, plus the confusion matrix. With imbalanced classes accuracy alone misleads, so the model is chosen on F1 or PR-AUC as stated in the training cell.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "RandomForestClassifier": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="RandomForestClassifier", dataset="telco_churn",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
labels = bundle.target_names or None
fig = viz.plot_confusion(y_test, pred, labels=labels)
save_figure(fig, NOTEBOOK_ID, "confusion"); plt.show()
if y_test.nunique() == 2:
    fig = viz.plot_roc_pr(final_model, X_test, y_test)
    save_figure(fig, NOTEBOOK_ID, "roc_pr"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()
# decision regions in two PCA axes
pca_view = PCA(n_components=2, random_state=SEED).fit(Xtr)
model_for_plot = clone(final_model.named_steps["model"])
fig = viz.plot_decision_boundary(lambda: clone(model_for_plot), pca_view.transform(Xtr), y_train.values,
                                 names=["PCA 1", "PCA 2"])
save_figure(fig, NOTEBOOK_ID, "decision_regions"); plt.show()

## 12. Interpretation

Permutation importance shows which inputs the model relies on. The confusion matrix shows which classes are mixed up, and the list of misclassified rows is the starting point for error analysis. A model is only as good as the cases it fails on.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring=SCORING, n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
fig = viz.plot_importance(importance.index, importance.values, title="Permutation importance on the test set",
                          top=12, xlabel="Drop in " + SCORING + " when shuffled")
save_figure(fig, NOTEBOOK_ID, "importance"); plt.show()
print("Top features:", ", ".join(importance.index[:5]))
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Out of bag score versus number of trees

OOB accuracy is computed without any validation split.

In [ ]:
rows = []
for n in [25, 50, 100, 200, 400]:
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=3, oob_score=True, n_jobs=2, random_state=SEED).fit(Xtr, y_train)
    rows.append({"trees": n, "oob_accuracy": m.oob_score_, "val_accuracy": m.score(Xva, y_val)})
display(pd.DataFrame(rows).set_index("trees").round(4))

### Experiment 2: Impurity versus permutation importance

Impurity importance is computed on training data and can favour continuous columns. Compare the rankings.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=2, random_state=SEED).fit(Xtr, y_train)
names = prep.get_feature_names_out()
imp_tree = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False).head(8)
perm_val = permutation_importance(rf, Xva, y_val, scoring="average_precision", n_repeats=5, random_state=SEED)
imp_perm = pd.Series(perm_val.importances_mean, index=names).sort_values(ascending=False).head(8)
display(pd.concat([imp_tree.rename("impurity (train)").reset_index(), imp_perm.rename("permutation (validation)").reset_index()], axis=1).round(4))

### Experiment 3: Forest versus single tree and class weights

Compare cross validated PR-AUC of a single tree, a forest and a balanced forest.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
candidates = {"single tree": DecisionTreeClassifier(min_samples_leaf=20, random_state=SEED),
              "forest": RandomForestClassifier(n_estimators=150, min_samples_leaf=3, n_jobs=2, random_state=SEED),
              "forest balanced": RandomForestClassifier(n_estimators=150, min_samples_leaf=3, class_weight="balanced", n_jobs=2, random_state=SEED)}
for name, m in candidates.items():
    s = cross_val_score(make_model_pipeline(X_trainval, m, scale=False), X_trainval, y_trainval, cv=5, scoring="average_precision")
    print(f"{name}: CV PR-AUC {s.mean():.3f} +- {s.std(ddof=1):.3f}")

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Change the metric used for model choice and see whether the chosen value changes.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Breiman shows that generalisation error depends on tree strength and correlation. Our bagging scratch cell shows the correlation effect numerically: averaging trees gives a higher validation AUC than a single tree.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Tune min_samples_leaf and max_features jointly, calibrate probabilities, and compare with gradient boosting under identical splits.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- Averaging trees reduces variance.
- OOB score is a free validation estimate.
- Permutation importance is safer than impurity importance.

**Use cases.** Strong default for tabular classification.

**Limitations.** Large models, importance caveats.

**Recommended next notebook.** Support Vector Machine.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.